In [1]:
import pandas as pd

df = pd.read_csv("novamarket_features_seleccionadas.csv")
df.head()

,nivel_lealtad_cod,precio_unitario,edad_cliente,monto_compra_log,canal_App,dias_desde_ultima_compra,total_gastado,fuga_cliente,id_cliente
0,3.0,475600.0,47.0,13.072335,False,340.0,832600.0,0,C33878
1,3.0,341900.0,27.0,12.742277,False,182.0,4296800.0,0,C87381
2,2.0,280900.0,45.0,12.545758,False,179.0,3045100.0,0,C34664
3,1.0,523400.0,63.0,13.168103,False,316.0,1721600.0,1,C106247
4,2.0,261600.0,51.0,13.167721,False,190.0,1488300.0,0,C60292


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['id_cliente','fuga_cliente'])
y = df['fuga_cliente']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

fuga_cliente
0    0.790889
1    0.209111
Name: proportion, dtype: float64
fuga_cliente
0    0.79089
1    0.20911
Name: proportion, dtype: float64


In [3]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Keep only numerical columns
numeric_cols = X_train.select_dtypes(include=['number']).columns

X_train_num = X_train[numeric_cols]
X_test_num = X_test[numeric_cols]

# 2. Scale the numerical data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

In [4]:
corr_matrix = X_train.corr(numeric_only=True)
print(corr_matrix)

                          nivel_lealtad_cod  precio_unitario  edad_cliente  \
nivel_lealtad_cod                  1.000000        -0.001483      0.000883   
precio_unitario                   -0.001483         1.000000     -0.002372   
edad_cliente                       0.000883        -0.002372      1.000000   
monto_compra_log                   0.003208         0.160555     -0.000719   
canal_App                         -0.001029         0.002230     -0.001298   
dias_desde_ultima_compra          -0.000739        -0.001076     -0.000262   
total_gastado                      0.001470         0.180522     -0.003161   

                          monto_compra_log  canal_App  \
nivel_lealtad_cod                 0.003208  -0.001029   
precio_unitario                   0.160555   0.002230   
edad_cliente                     -0.000719  -0.001298   
monto_compra_log                  1.000000   0.000255   
canal_App                         0.000255   1.000000   
dias_desde_ultima_compra         

## Correlación notable (<0.6)

Ninguna. No se necesita PCA

In [5]:
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

fuga_cliente
0    316105
1     83578
Name: count, dtype: int64
fuga_cliente
0    0.790889
1    0.209111
Name: proportion, dtype: float64


In [6]:
!pip install --upgrade imbalanced-learn

In [7]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

In [8]:
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print(y_train_under.value_counts())

fuga_cliente
0    83578
1    83578
Name: count, dtype: int64


In [9]:
ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

print(y_train_over.value_counts())

fuga_cliente
1    316105
0    316105
Name: count, dtype: int64


In [10]:
X_train.dtypes

nivel_lealtad_cod           float64
precio_unitario             float64
edad_cliente                float64
monto_compra_log            float64
canal_App                      bool
dias_desde_ultima_compra    float64
total_gastado               float64
dtype: object

In [13]:
from sklearn.impute import SimpleImputer

# 1. Fill missing numerical values with the mean of each column
imputer = SimpleImputer(strategy='mean') # or strategy='median'

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

smote = SMOTE(random_state=42)
# 2. Now run SMOTE on the imputed data
X_train_smote, y_train_smote = smote.fit_resample(X_train_imputed, y_train)

print(y_train_smote.value_counts())

fuga_cliente
1    316105
0    316105
Name: count, dtype: int64


In [14]:
train_df = pd.DataFrame(X_train_smote) 

train_df['fuga_cliente'] = y_train_smote

train_df.to_csv('novamarket_train_balanceado.csv', index=False)

test_df = pd.DataFrame(X_test)

test_df['fuga_cliente'] = y_test.values if hasattr(y_test, 'values') else y_test

test_df.to_csv('novamarket_test.csv', index=False)

print("¡Archivos guardados con éxito!")

¡Archivos guardados con éxito!
